In [24]:
from storyforge.config import load_config
from storyforge.data import prepare_stories
from storyforge.tokenizer import train_tokenizer
from storyforge.encoding import encode_stream
from pathlib import Path
import numpy as np

In [12]:
%cd /Users/akarsh/Documents/projects/fastInfer/StoryForge

/Users/akarsh/Documents/projects/fastInfer/StoryForge


In [13]:
config = load_config("configs/default.toml")

In [14]:
train_texts, val_texts = prepare_stories(config)

In [15]:
print("Training stories:", len(train_texts))
print("Validation stories:", len(val_texts))

Training stories: 100000
Validation stories: 2000


In [16]:
tokenizer = train_tokenizer(train_texts, config)

In [17]:
print("Vocabulary size:", tokenizer.get_vocab_size())
print("Saved to:", config["tokenizer"]["path"])

Vocabulary size: 8192
Saved to: artifacts/tokenizer.json


In [20]:
settings = config["encoding"]

In [21]:
eos_id = tokenizer.token_to_id(
    config["tokenizer"]["eos_token"]
)

In [22]:
train_tokens = encode_stream(
    train_texts,
    tokenizer,
    eos_id,
    settings["batch_size"],
)

val_tokens = encode_stream(
    val_texts,
    tokenizer,
    eos_id,
    settings["batch_size"],
)

In [23]:
print("Training tokens:", len(train_tokens))
print("Validation tokens:", len(val_tokens))
print("Storage type:", train_tokens.dtype)

Training tokens: 21611495
Validation tokens: 382835
Storage type: uint16


In [25]:
train_path = Path(settings["train_path"])
val_path = Path(settings["validation_path"])

In [26]:
train_path.parent.mkdir(parents=True, exist_ok=True)
val_path.parent.mkdir(parents=True, exist_ok=True)

In [27]:
np.save(train_path, train_tokens)
np.save(val_path, val_tokens)